In [ ]:
%pip install -r requirements.txt

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/sample_submission.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/dataset-metadata.json
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/baseline_submission.csv
/kaggle/input/models/israelolawuyi/baai-bge-base-finetune-agric-doctri-ai/transformers/default/1/__huggingface_repos__.json
/kaggle/input/models/israelolawuyi/baai-bge-base-finetune-agric-doctri-ai/transformers/default/1/fine_tuned_bge_base_agri/config.json
/kaggle/input/models/israelolawuyi/

In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv", index_col ="document_id")
test = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv")
df.head()
test.head()

,query_id,query
0,1001,How do I cope with drought and erratic rainfal...
1,1002,How can I adapt my farming to drought and erra...
2,1003,How does drought and erratic rainfall affect m...
3,1004,What is the risk of drought and erratic rainfa...
4,1005,How do I cope with heat stress on my farm?


In [4]:
# My plan is to use hybird method which involves
# bm25 + dense retrival + reranker
def create_search_content(row):
    title = str(row.get("title", " ")).strip()
    text = str(row.get("text", " ")).strip()
    source = str(row.get("source", " ")).strip()
    crop = str(row.get("crop", " ")).strip()
    country = str(row.get("country", " ")).strip()
    source_url = str(row.get("source_url", " ")).strip()
    
    return f"Crop {crop} | Country {country} | Title {title} | Text {text} | Source {source}"
df_upd=pd.DataFrame()
df["search_text"] = df.apply(create_search_content, axis = 1)
docs_ids = df.index.to_list()
corpus_texts = df['search_text'].tolist()

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
##import torch.nn as nn
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tqdm.auto import tqdm

# ----------------------------------------------------
# 1. Device Setup & Load Models
# ----------------------------------------------------
num_gpus = torch.cuda.device_count()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 1a. Dense Bi-Encoder 
from sentence_transformers import SentenceTransformer

dense_model = SentenceTransformer("/kaggle/input/models/israelolawuyi/baai-bge-base-finetune-agric-doctri-ai/transformers/default/1/fine_tuned_bge_base_agri", device=device)

# 1b. Fine-Tuned BGE Reranker Large from local disk
save_dir = "/kaggle/input/models/israelolawuyi/bge-reranker-largetri-ai-agri/transformers/default/1/fine_tuned_bge_reranker"
reranker = CrossEncoder(save_dir, max_length=512, device=device)
##if num_gpus > 1:
    ##reranker.model = nn.DataParallel(reranker.model)
# ----------------------------------------------------
# 2. Setup Document Corpus & First-Stage Indexing
# ----------------------------------------------------
# Assumes docs_ids and corpus_texts are already populated
id_to_text = dict(zip(docs_ids, corpus_texts))
num_docs = len(corpus_texts)

# 2a. BM25 Indexing
tokenized_corpus = [doc.lower().split() for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

# 2b. Dense Indexing (pre-encode all documents once)
print("Encoding corpus passages with dense bi-encoder...")
doc_embeddings = dense_model.encode(
    corpus_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_tensor=True,
    device=device,
)

# ----------------------------------------------------
# 3. First-Stage Retrieval Functions
# ----------------------------------------------------
def search_bm25(query_text, top_k=50):
    tokenized_query = query_text.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [docs_ids[i] for i in top_indices]

def search_dense(query_text, top_k=50):
  prefixed_query = (
      f"Represent this sentence for searching relevant passages: {query_text}"
  )
  q_emb = dense_model.encode(
      prefixed_query,
      normalize_embeddings=True,
      convert_to_tensor=True,
      device=device,
  )

  # Cosine similarity via dot product on normalized vectors
  sim_scores = torch.matmul(doc_embeddings, q_emb)
  top_res = torch.topk(sim_scores, k=min(top_k, num_docs))

  indices = top_res.indices.cpu().numpy()
  scores = top_res.values.cpu().numpy()

  # Return dict of {doc_id: cosine_sim} and list of IDs
  dense_scores_dict = {
      docs_ids[i]: float(scores[idx]) for idx, i in enumerate(indices)
  }
  return list(dense_scores_dict.keys()), dense_scores_dict

def reciprocal_rank_fusion(bm25_ids, dense_ids, k=60):
    rrf_scores = {}
    for rank, doc_id in enumerate(bm25_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    for rank, doc_id in enumerate(dense_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in sorted_docs]

# ----------------------------------------------------
# 4. Hybrid Retrieval + Cross-Encoder Reranking
# ----------------------------------------------------
# ----------------------------------------------------
# 4. Hybrid Retrieval + Alpha=0.0 Score Blending
# ----------------------------------------------------
CANDIDATE_POOL_SIZE = 50
ALPHA = 0.0  # 0.0% Dense Cosine Sim, 100% Normalized Reranker Logits

query_id_col = (
    "QueryId"
    if "QueryId" in test.columns
    else ("query_id" if "query_id" in test.columns else test.columns[0])
)
query_text_col = (
    "Query"
    if "Query" in test.columns
    else ("query" if "query" in test.columns else test.columns[1])
)

submission_rows = []

print("Running Hybrid Retrieval with Alpha=0.0 Score Blending...")
for _, row in tqdm(test.iterrows(), total=len(test)):
  q_id = str(row[query_id_col]).replace(".0", "")
  q_text = str(row[query_text_col]).strip()

  # Step 1: Candidate collection
  bm25_top = search_bm25(q_text, top_k=CANDIDATE_POOL_SIZE)
  dense_top_ids, dense_score_map = search_dense(
      q_text, top_k=CANDIDATE_POOL_SIZE
  )

  # Step 2: Merge candidates via RRF to capture top-50 pool
  candidate_ids = reciprocal_rank_fusion(bm25_top, dense_top_ids, k=60)[
      :CANDIDATE_POOL_SIZE
  ]

  # Step 3: Score candidates with fine-tuned cross-encoder
  pairs = [[q_text, id_to_text[doc_id]] for doc_id in candidate_ids]
  raw_reranker_scores = reranker.predict(
      pairs, batch_size=32, convert_to_numpy=True
  )

  # Step 4: Alpha Blending (Alpha=0.0)
  # Sigmoid maps unbounded logits to [0, 1]
  norm_reranker = 1.0 / (1.0 + np.exp(-raw_reranker_scores))

  blended_scores = []
  for idx, doc_id in enumerate(candidate_ids):
    # If a document came via BM25 only, its dense similarity is defaulted to 0.0
    d_score = dense_score_map.get(doc_id, 0.0)
    r_score = norm_reranker[idx]
    blended = (ALPHA * d_score) + ((1.0 - ALPHA) * r_score)
    blended_scores.append(blended)

  blended_scores = np.array(blended_scores)

  # Step 5: Pick the final Top 5 highest-confidence documents
  top_5_indices = np.argsort(blended_scores)[::-1][:5]
  top_5_ids = [candidate_ids[idx] for idx in top_5_indices]

  for doc_id in top_5_ids:
    submission_rows.append({"QueryId": q_id, "DocumentId": str(doc_id)})

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Encoding corpus passages with dense bi-encoder...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Running Hybrid Retrieval with Alpha=0.2 Score Blending...


  0%|          | 0/200 [00:00<?, ?it/s]

In [6]:
# ----------------------------------------------------
# 5. Validation & Export
# ----------------------------------------------------
sub_df = pd.DataFrame(submission_rows)

assert list(sub_df.columns) == ["QueryId", "DocumentId"], "Header mismatch!"
assert len(sub_df) == len(test) * 5, f"Expected {len(test) * 5} rows, got {len(sub_df)}"
assert (
    sub_df.groupby("QueryId")["DocumentId"].nunique() == 5
).all(), "Duplicate documents found in top 5!"

sub_df.to_csv("submission_hybrid_finetuned_reranker.csv", index=False)
print("Saved 'submission_hybrid_finetuned_reranker.csv' successfully!")
print(sub_df.head(10))

Saved 'submission_hybrid_finetuned_reranker.csv' successfully!
  QueryId DocumentId
0    1001          2
1    1001          3
2    1001          4
3    1001          5
4    1001          1
5    1002          2
6    1002          5
7    1002          3
8    1002          4
9    1002          1
